# Week 2 — Implement and Verify the VAE Loss

**Course:** Noise → Masterpiece (Build Stable Diffusion from First Principles) — Week 2 (graded)

We maximize `log p(x)`, but it's an intractable integral over all latent codes. Introducing an encoder
`q(z|x)` gives a tractable lower bound, the **ELBO**, whose negative is the loss:

$$\mathcal{L} = \underbrace{\text{BCE}(\hat{x},x)}_{\text{reconstruction}} \;+\; \underbrace{-\tfrac12\sum\left(1+\log\sigma^2-\mu^2-\sigma^2\right)}_{\text{KL}(q(z|x)\,\|\,N(0,I))}$$

This notebook implements both terms from scratch, verifies them against `torch.distributions`, and answers
the reflection questions.

## Setup

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

## Part 1 — Closed-form KL, from scratch (30 pts)

For `q = N(μ, σ²)` and prior `p = N(0, 1)`, the KL has a closed form. Plugging both Gaussian densities into
the KL integral and using `E[z]=μ`, `E[z²]=μ²+σ²` gives, per dimension:

$$\text{KL} = \tfrac12\left(\mu^2 + \sigma^2 - 1 - \log\sigma^2\right) = -\tfrac12\left(1 + \log\sigma^2 - \mu^2 - \sigma^2\right)$$

We work in `logvar = log σ²`, so `σ² = exp(logvar)`. Sum over all dims and the batch to get a scalar.

In [ ]:
def kl_divergence_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    \"\"\"
    Closed-form KL divergence between N(mu, exp(logvar)) and N(0, 1).
    Returns a scalar (summed over all dimensions and batch).
    \"\"\"
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())


# Test it
mu     = torch.randn(32, 16)   # batch of 32, latent dim 16
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("✅ Part 1 passed")

## Part 2 — Reparameterization from scratch (20 pts)

We can't backprop through a random sample, so instead of drawing `z ~ N(μ, σ²)` directly we draw noise
`ε ~ N(0, I)` and compute `z = μ + σ·ε`. The randomness now lives in `ε` (no parameters), so `z` is a
differentiable function of `μ` and `σ`. Note `std = exp(0.5 · logvar)` since `logvar = log σ²`.

In [ ]:
def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    \"\"\"
    Sample z using the reparameterization trick.
    z = mu + std * eps, where eps ~ N(0, I)
    \"\"\"
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + std * eps


# Verify: the mean of many samples should be close to mu
mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
print("✅ Part 2 passed")

## Part 3 — Full ELBO loss (30 pts)

Combine the two terms. The reconstruction term is **binary cross-entropy** here (the assignment assumes pixel
values in `[0, 1]`, i.e. a Bernoulli likelihood). Both terms use `reduction='sum'` so they stay on the same
scale; the KL reuses Part 1.

In [ ]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor) -> torch.Tensor:
    \"\"\"
    ELBO loss = Reconstruction loss + KL divergence
    - Reconstruction: BCE between x_recon and x, summed over pixels and batch
    - KL: closed-form KL between N(mu, exp(logvar)) and N(0, I)
    Returns a scalar.
    \"\"\"
    recon = F.binary_cross_entropy(x_recon, x, reduction='sum')
    kl = kl_divergence_gaussian(mu, logvar)
    return recon + kl


# Quick smoke test
batch, dim, latent = 16, 784, 32
x       = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu      = torch.randn(batch, latent)
logvar  = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("✅ Part 3 passed")

## Part 4 — Analysis & reflection (20 pts)

**Numerical check for Q4** (run `elbo_loss`'s KL term with `mu` and `logvar` all zeros):

In [ ]:
mu_zero     = torch.zeros(16, 32)
logvar_zero = torch.zeros(16, 32)
kl_zero = kl_divergence_gaussian(mu_zero, logvar_zero)
print(f"KL term with mu=0, logvar=0: {kl_zero.item():.6f}")

**Q1 — Why can't we just maximise `log p(x)` directly?**

`log p(x) = log ∫ p(x|z) p(z) dz` integrates the decoder's output over the *entire* latent space. For any
non-trivial latent dimension this integral has no closed form, so we can't even evaluate the objective, let
alone differentiate it. The encoder `q(z|x)` gets around this: instead of integrating over every possible `z`,
it proposes the few `z`'s that are actually plausible for a given `x`. That turns the intractable integral into
a tractable expectation we can estimate by sampling, and gives us the ELBO — a lower bound on `log p(x)` that we
*can* optimize.

**Q2 — What happens if you remove the KL term?**

It collapses into an ordinary autoencoder. With nothing pulling the encoder toward `N(0, I)`, it's free to drive
`σ → 0` and place each input's `μ` wherever minimizes reconstruction. Reconstruction would look excellent, but
the latent space becomes a scatter of isolated points with empty gaps between them. At generation time we sample
`z ~ N(0, I)` and land in those gaps — regions the decoder never saw — so generation produces garbage. The KL
term is exactly what keeps the latent codes packed and overlapping so the space stays smooth and sampleable.

**Q3 — Why does the reparameterization trick work?**

Drawing `z ~ N(μ, σ²)` is a stochastic node: there is no differentiable path from the loss back to `μ` and `σ`
through "sample a random number," so gradients can't flow. Rewriting `z = μ + σ·ε` with `ε ~ N(0, I)` moves all
the randomness into `ε`, which carries no parameters. Now `z` is a deterministic, differentiable function of `μ`
and `σ`, so the gradient flows from the loss through `z` into `μ` and `σ`, and onward into the encoder weights.
Without the trick the gradient hits the RNG and stops — you'd be forced to use a high-variance estimator like
REINFORCE instead.

**Q4 — `elbo_loss` with `mu` all zeros and `logvar` all zeros: what is the KL term?**

It is **0** (the cell above prints `0.000000`). With `μ = 0` and `logvar = 0`, we have `σ² = exp(0) = 1`, so the
encoder distribution `q = N(0, 1)` is *identical* to the prior `p = N(0, 1)`, and the KL between identical
distributions is zero. The closed form agrees: `-0.5 · Σ(1 + 0 - 0² - exp(0)) = -0.5 · Σ(1 + 0 - 0 - 1) = 0`. It
matches because the formula is nothing more than the KL between the encoder Gaussian and the standard-normal
prior — and here they are the same Gaussian.